In [1]:
!pip install -q transformers peft trl bitsandbytes accelerate datasets wandb huggingface_hub groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 11.0 MB/s eta 0:00:00


In [ ]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from peft import PeftModel
from groq import Groq
import time
import os


# ── load BASE model + ADAPTER ─────────────────────────────────────────────────
BASE_MODEL = "microsoft/Phi-3.5-mini-instruct"
ADAPTER_HF_ID = "TeslaInch/scd-phi35-adapter-v8"
ADAPTER_NAME = "scd-phi35-adapter-v8"  

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("Loading Phi-3.5 Mini BASE model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16,
    attn_implementation="eager",
)
#base_model.eval()

print(f"Loading adapter from Hugging Face Hub: {ADAPTER_HF_ID}")
model = PeftModel.from_pretrained(base_model, ADAPTER_HF_ID)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Adapter loaded: {ADAPTER_NAME}")
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# ── load files 
with open("/kaggle/input/datasets/teslaincarnate/jsonfiles/all_questions.json") as f:  
    questions = json.load(f)

with open("/kaggle/input/datasets/teslaincarnate/jsonfiles/all_answers.json") as f:  
    answers = json.load(f)

answer_map = {a["id"]: a for a in answers}

case_map={}

print(f"Questions: {len(questions)}")
print(f"Answers: {len(answers)}")

# ── inference pipeline 
print("\nSetting up inference pipeline...")
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=400,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
)

SYSTEM = (
    "You are a medical AI assistant specialised in sickle cell disease. "
    "Answer the following clinical question accurately and completely."
)

def build_prompt(q):
    case = case_map.get(q["id"], "")
    if case:
        user_content = f"{SYSTEM}\nClinical case:\n{case}\n\nQuestion: {q['question']}"
    else:
        user_content = f"{SYSTEM}\n{q['question']}"
    return f"<|user|>\n{user_content}<|end|>\n<|assistant|>\n"

def get_response(q):
    prompt = build_prompt(q)
    output = pipe(prompt)
    full_text = output[0]["generated_text"]
    response = full_text.split("<|assistant|>")[-1].strip()
    response = response.replace("<|end|>", "").strip()
    return response

# ── groq judge 
groq_client = Groq(api_key="Your_api_key_here")

LAYER_RUBRICS = {
    1: "Score 0-2. 2=correct answer with sound reasoning. 1=partially correct or incomplete. 0=wrong or dangerous.",
    2: "Score 0-1. 1=correct answer. 0=wrong answer.",
    3: "Score 0-2. 2=correct, matches guideline. 1=partially correct. 0=wrong or contradicts guideline.",
    4: "Score 0-3. 3=correct diagnosis and full management in priority order. 2=correct diagnosis, partial management. 1=partial diagnosis or incomplete management. 0=wrong or dangerous.",
    5: "Score 0-2. 2=correct, matches ASH guideline. 1=partially correct. 0=wrong or contradicts guideline.",
}

MAX_SCORES = {1: 2, 2: 1, 3: 2, 4: 3, 5: 2}

def judge(question_text, gold_answer, model_response, layer, case=""):
    full_q = f"Clinical case:\n{case}\n\nQuestion: {question_text}" if case else question_text
    rubric = LAYER_RUBRICS.get(layer, LAYER_RUBRICS[1])

    prompt = f"""You are a clinical expert evaluating a medical AI response on sickle cell disease.

QUESTION: {full_q}

GOLD STANDARD ANSWER: {gold_answer}

MODEL RESPONSE: {model_response}

RUBRIC: {rubric}

Respond in this exact JSON format with no other text:
{{"score": <int>, "reasoning": "<one sentence>"}}"""

    for attempt in range(5):
        try:
            response = groq_client.chat.completions.create(
                model="llama-3.1-8b-instant",
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
            )
            text = response.choices[0].message.content.strip()
            if "```" in text:
                text = text.split("```")[1]
                if text.startswith("json"):
                    text = text[4:]
            parsed = json.loads(text.strip())
            return {
                "score": int(parsed["score"]),
                "judge_reasoning": parsed["reasoning"],
                "judge_error": False
            }
        except Exception as e:
            if "429" in str(e) or "rate" in str(e).lower():
                wait = (2 ** attempt) * 3
                print(f"  Rate limited. Waiting {wait}s...")
                time.sleep(wait)
            else:
                return {
                    "score": None,
                    "judge_reasoning": f"Error: {str(e)}",
                    "judge_error": True
                }

    return {"score": None, "judge_reasoning": "Max retries exceeded", "judge_error": True}

# ── main eval loop 
print("\nStarting ADAPTER EVALUATION...")
print("Checkpoint saved every 50 questions\n")

results = []
OUTPUT_PATH = "/kaggle/working/adapter_eval_results.json"

for i, q in enumerate(questions):
    qid = q["id"]
    layer = q["layer"]

    if qid not in answer_map:
        print(f"  [{i+1}] {qid} — no answer found, skipping")
        continue

    gold = answer_map[qid]["answer"]
    case = case_map.get(qid, "")

    print(f"[{i+1}/{len(questions)}] {qid}...", end=" ", flush=True)

    try:
        response = get_response(q)
    except Exception as e:
        print(f"inference error: {e}")
        continue

    judgment = judge(q["question"], gold, response, layer, case)
    max_score = MAX_SCORES.get(layer, 2)

    result = {
        "id": qid,
        "layer": layer,
        "category": q.get("category", ""),
        "source": q.get("source", ""),
        "question": q["question"],
        "gold_answer": gold,
        "model_response": response,
        "score": judgment["score"],
        "max_score": max_score,
        "pct_score": round(judgment["score"] / max_score * 100, 1) if judgment["score"] is not None else None,
        "judge_reasoning": judgment["judge_reasoning"],
        "judge_error": judgment["judge_error"],
        "model": f"phi35-mini-adapter-{ADAPTER_NAME}"
    }
    results.append(result)

    score_str = f"{judgment['score']}/{max_score}" if judgment["score"] is not None else "error"
    print(f"score: {score_str}")

    time.sleep(2)

    if (i + 1) % 50 == 0:
        with open(OUTPUT_PATH, "w") as f:
            json.dump(results, f, indent=2)
        print(f"\n  [checkpoint saved — {i+1}/{len(questions)} done]\n")

# ── final save ────────────────────────────────────────────────────────────────
with open(OUTPUT_PATH, "w") as f:
    json.dump(results, f, indent=2)

print(f"\nDone. Results saved to {OUTPUT_PATH}")

# ── summary ───────────────────────────────────────────────────────────────────
scored = [r for r in results if r["score"] is not None]
pct_scores = [r["score"] / r["max_score"] for r in scored]
overall = sum(pct_scores) / len(pct_scores) * 100

print(f"\n{'='*50}")
print(f"ADAPTER EVAL RESULTS: {ADAPTER_NAME}")
print(f"{'='*50}")
print(f"Scored:  {len(scored)}/{len(results)}")
print(f"Overall: {overall:.1f}%")

print(f"\nBy layer:")
for layer in sorted(set(r["layer"] for r in scored)):
    lr = [r for r in scored if r["layer"] == layer]
    lpct = sum(r["score"]/r["max_score"] for r in lr) / len(lr) * 100
    print(f"  Layer {layer}: {lpct:.1f}%  ({len(lr)} questions)")

print(f"\nBy category:")
cats = sorted(set(r["category"] for r in scored if r["category"]))
for cat in cats:
    cr = [r for r in scored if r["category"] == cat]
    cpct = sum(r["score"]/r["max_score"] for r in cr) / len(cr) * 100
    print(f"  {cat:<35} {cpct:.1f}%  ({len(cr)} questions)")

Loading Phi-3.5 Mini BASE model...


config.json: 0.00B [00:00, ?B/s]

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

score: 2/2
[583/584] layer5_007... 

Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


score: 2/2
[584/584] layer5_008... 

Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


score: 2/2

Done. Results saved to /kaggle/working/adapter_eval_results.json

ADAPTER EVAL RESULTS: scd-phi35-adapter-v8
Scored:  576/584
Overall: 69.8%

By layer:
  Layer 1: 67.3%  (49 questions)
  Layer 2: 71.2%  (451 questions)
  Layer 3: 51.3%  (39 questions)
  Layer 4: 74.7%  (29 questions)
  Layer 5: 81.2%  (8 questions)

By category:
  contraindication                    41.7%  (8 questions)
  diagnosis_criteria                  80.8%  (13 questions)
  dosing_guideline                    65.1%  (21 questions)
  emergency_management                54.4%  (19 questions)
  multiple_choice                     71.5%  (449 questions)
  pathophysiology                     81.2%  (8 questions)
  screening_schedule                  65.8%  (20 questions)
  treatment_threshold                 65.7%  (36 questions)
  yes_no_maybe                        0.0%  (2 questions)
